In [1]:
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
import os
import liana as li
from liana.method import cellphonedb, cellchat
from tqdm import tqdm
#liana dotplot returns a ggsave object...
from plotnine import ggsave, ggplot
import numpy as np
import warnings
warnings.filterwarnings("ignore")

In [2]:
folder = "IndividualMiceData/"
Mice = ["GF_HF_6B","GF_HF_13L","SPF_HF_118","SPF_HF_131","SPF_HF_133","GF_HFVHC_5A","GF_HFVHC_16L","SPF_HFVHC_124","SPF_HFVHC_136","SPF_HFVHC_137"]
method = ["CellPhoneDB", "CellChat"]
def get_data():
    data = {}
    for mouse in tqdm(Mice):
        data[mouse] = {}
        for m in method:
            file = folder+mouse+"_"+m+".h5ad"
            data[mouse][m] = sc.read_h5ad(file)
    return data
data = get_data()

100%|██████████████████████████████████████████████████████████████████████████████████████| 10/10 [00:24<00:00,  2.43s/it]


In [23]:
# st = ['Endothelial cells -> Neutrophils','Kupffer cells -> B cells','pDCs -> ILC1s']
# lr = ['Apoe -> Sorl1','Cfh -> Itgam','Tgfb1 -> Cxcr4']
def AnalysisOnOneMouse(data,index):
    mouse_curr = list(data.keys())[index]
    mouse_curr_data = {}
    for m in method:
        x = data[mouse_curr][m].uns["cpdb_res"]
        if m == 'CellPhoneDB':
            x = x[x['lr_means'] != 0]
            x = x[x['cellphone_pvals'] < 0.05]
        else:
            x = x[x['lr_probs'] != 0]
            x = x[x['cellchat_pvals'] < 0.05]
        x = x[['ligand', 'receptor', 'source','target']]
        x['Source -> Target'] = x['source'] + ' -> ' + x['target']
        x['Ligand -> Receptor'] = x['ligand'] + ' -> ' + x['receptor'] 
        # x = x[x['Source -> Target'].isin(st)]
        # x = x[x['Ligand -> Receptor'].isin(lr)]
        mouse_curr_data[m] = x[['Source -> Target', 'Ligand -> Receptor']]

    # Group by 'Source -> Target' in df1 and get unique 'Ligand -> Receptor' counts
    df1_grouped = mouse_curr_data[method[0]].groupby('Source -> Target')['Ligand -> Receptor'].nunique().reset_index(name=method[0])
    
    # Group by 'Source -> Target' in df2 and get unique 'Ligand -> Receptor' counts
    df2_grouped = mouse_curr_data[method[1]].groupby('Source -> Target')['Ligand -> Receptor'].nunique().reset_index(name=method[1])
    
    # Merge the grouped DataFrames on 'Source -> Target'
    df_merged = pd.merge(df1_grouped, df2_grouped, on='Source -> Target', how='outer').fillna(0)
    
    # Function to find common and different 'Ligand -> Receptor' within each 'Source -> Target' group
    def analyze_lr(group1, group2):
        set1 = set(group1)
        set2 = set(group2)
        common = len(set1.intersection(set2))
        diff1 = len(set1 - set2)
        diff2 = len(set2 - set1)
        a = 'Different in ' + method[0]
        b = 'Different in ' + method[1]
        return pd.Series({'Common': common, a: diff1, b: diff2})
    
    # Group by 'Source -> Target' and apply the analysis function
    df_analysis = pd.concat([mouse_curr_data[method[0]].groupby('Source -> Target')['Ligand -> Receptor'].unique().rename('LRs DF1'),
                             mouse_curr_data[method[1]].groupby('Source -> Target')['Ligand -> Receptor'].unique().rename('LRs DF2')], axis=1).fillna('').apply(lambda x: analyze_lr(x['LRs DF1'], x['LRs DF2']), axis=1).reset_index()
    
    # Final Merged DataFrame
    df_new = pd.merge(df_merged, df_analysis, on='Source -> Target', how='outer').fillna(0)
    # file_name = "Result_tables/"+mouse_curr+".csv"
    # df_new.to_csv(file_name, index=False)
    print(mouse_curr)
    return df_new

In [24]:
d = {}
for i in range(10):
    m = mouse_curr = list(data.keys())[i]
    d[m] = AnalysisOnOneMouse(data,i)

GF_HF_6B
GF_HF_13L
SPF_HF_118
SPF_HF_131
SPF_HF_133
GF_HFVHC_5A
GF_HFVHC_16L
SPF_HFVHC_124
SPF_HFVHC_136
SPF_HFVHC_137


In [26]:
for df in d.values():
    print(df['Common'])

0    3
1    2
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64
0    3
1    2
2    0
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64
0    3
1    2
2    0
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64
0    3
1    2
2    0
Name: Common, dtype: int64
0    3
1    2
Name: Common, dtype: int64


In [22]:
# Extract the 's' column from each DataFrame and convert to a set
sets_of_s = [set(df['CellChat']['Ligand -> Receptor']) for df in d.values()]

# Find the intersection of all the sets
common_values = set.intersection(*sets_of_s)

common_values

{'Actr2 -> Adrb2',
 'Adam10 -> Cd44',
 'Adam10 -> Il6ra',
 'Adam10 -> Notch2',
 'Adam11 -> Itga4',
 'Adam17 -> Il6ra',
 'Adam17 -> Rhbdf2',
 'Adam23 -> Itga4',
 'Anxa2 -> Tlr2',
 'Apoe -> Abca1',
 'Apoe -> Sorl1',
 'App -> Cd74',
 'App -> Fpr2',
 'App -> Notch2',
 'App -> Rpsa',
 'Arpc5 -> Adrb2',
 'B2m -> Klrc1',
 'B2m -> Klrd1',
 'Bgn -> Tlr2',
 'Bgn -> Tlr4',
 'Bmp2 -> Actr2',
 'C1qb -> C1qbp',
 'Calr -> Itgav',
 'Ccl4 -> Ccr5',
 'Cd14 -> Itga4',
 'Cd14 -> Itgb1',
 'Cd200 -> Cd200r1',
 'Cd200 -> Cd200r2',
 'Cd200 -> Cd200r4',
 'Cd47 -> Sirpa',
 'Cd48 -> Cd2',
 'Cfh -> Itgam',
 'Cfp -> Ncr1',
 'Col4a1 -> Cd44',
 'Col4a1 -> Cd47',
 'Col4a2 -> Cd44',
 'Copa -> Cd74',
 'Dll4 -> Notch1',
 'Dll4 -> Notch2',
 'Fam3c -> Lamp1',
 'Gnai2 -> Adcy7',
 'Gnai2 -> C5ar1',
 'Gnai2 -> Ccr5',
 'Gnai2 -> Cxcr2',
 'Gnai2 -> Cxcr3',
 'Gnai2 -> Igf1r',
 'Gnai2 -> Oprm1',
 'Gnai2 -> S1pr1',
 'Gnai2 -> S1pr4',
 'Gnas -> Adcy7',
 'Gnas -> Adrb2',
 'Grn -> Tnfrsf1a',
 'Grn -> Tnfrsf1b',
 'Gstp1 -> Traf2',
 '

In [ ]:
st = ['Endothelial cells -> Neutrophils','Kupffer cells -> B cells','pDCs -> ILC1s']

In [29]:
data[list(data.keys())[0]][method[1]].uns["cpdb_res"].columns

Index(['ligand', 'ligand_complex', 'ligand_props', 'ligand_trimean', 'mat_max',
       'receptor', 'receptor_complex', 'receptor_props', 'receptor_trimean',
       'source', 'target', 'lr_probs', 'cellchat_pvals'],
      dtype='object')